# MediTwin AI — Patient Risk EDA (Phase 2)

**Educational / synthetic data only.** Risk labels are rule-based synthetic tags,
NOT a clinical decision system.

Run the pipeline first if processed files are missing:

```bash
python -m src.features.feature_engineering
```

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.preprocess import load_patient_data, preprocess_patient_data
from src.features.feature_engineering import engineer_patient_features

pd.set_option("display.max_columns", 40)
print("Project root:", PROJECT_ROOT)

## 1. Load processed patient features

In [ ]:
feat_path = PROJECT_ROOT / "data" / "processed" / "patient_data_features.csv"

if feat_path.is_file():
    df = pd.read_csv(feat_path)
    print("Loaded", feat_path)
else:
    clean, report = preprocess_patient_data(load_patient_data())
    report.print_report()
    df = engineer_patient_features(clean)

print(df.shape)
assert "risk_level" in df.columns, "Target risk_level must be preserved"
df.head()

## 2. Risk class distribution

In [ ]:
order = ["Low", "Medium", "High"]
counts = df["risk_level"].value_counts().reindex(order)
print(counts)
print("\nProportions (%):")
print((100 * counts / counts.sum()).round(1))

fig = px.bar(x=counts.index, y=counts.values, text=counts.values,
             labels={"x": "risk_level", "y": "count"},
             title="Patient risk class distribution (synthetic)",
             color=counts.index,
             color_discrete_map={"Low": "#2ca02c", "Medium": "#ff7f0e", "High": "#d62728"})
fig.update_traces(textposition="outside")
fig.show()

## 3. Age vs risk level

In [ ]:
fig = px.box(df, x="risk_level", y="age", category_orders={"risk_level": order},
             color="risk_level",
             color_discrete_map={"Low": "#2ca02c", "Medium": "#ff7f0e", "High": "#d62728"},
             title="Age distribution by synthetic risk level")
fig.show()

print(df.groupby("risk_level")["age"].describe().reindex(order).round(1))

In [ ]:
if "age_group" in df.columns:
    ct = pd.crosstab(df["age_group"], df["risk_level"])[order]
    display(ct)
    fig = px.imshow(ct, text_auto=True, aspect="auto",
                    title="Age group vs risk level counts",
                    color_continuous_scale="YlOrRd")
    fig.show()

## 4. Vital signs distribution

In [ ]:
vital_cols = ["heart_rate", "systolic_bp", "diastolic_bp",
              "temperature", "oxygen_saturation"]

fig = px.histogram(df, x="heart_rate", color="risk_level", barmode="overlay",
                   opacity=0.65, category_orders={"risk_level": order},
                   title="Heart rate by risk level")
fig.show()

fig = px.histogram(df, x="oxygen_saturation", color="risk_level", barmode="overlay",
                   opacity=0.65, category_orders={"risk_level": order},
                   title="Oxygen saturation by risk level")
fig.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.flatten()
for i, col in enumerate(vital_cols):
    axes[i].hist(df[col], bins=30, color="steelblue", edgecolor="white")
    axes[i].set_title(col)
axes[-1].axis("off")
fig.suptitle("Vital signs — overall distributions (Matplotlib)")
plt.tight_layout()
plt.show()

display(df[vital_cols].describe().round(2))

## 5. Vitals vs risk (box plots)

In [ ]:
for col in ["systolic_bp", "temperature", "oxygen_saturation"]:
    fig = px.box(df, x="risk_level", y=col, category_orders={"risk_level": order},
                 color="risk_level",
                 color_discrete_map={"Low": "#2ca02c", "Medium": "#ff7f0e", "High": "#d62728"},
                 title=f"{col} by risk level")
    fig.show()

## 6. Clinical flags vs risk

In [ ]:
for col in ["chronic_disease", "emergency_case", "gender"]:
    ct = pd.crosstab(df[col], df["risk_level"], normalize="index")[order] * 100
    print(f"\n{col} → risk_level row %")
    display(ct.round(1))
    fig = px.histogram(df, x=col, color="risk_level", barmode="group",
                       category_orders={"risk_level": order},
                       title=f"{col} vs risk_level counts")
    fig.show()

## 7. Correlation among numeric inputs (exclude target encoding)

In [ ]:
num_cols = ["age", "heart_rate", "systolic_bp", "diastolic_bp",
            "temperature", "oxygen_saturation", "previous_admissions"]
if "pulse_pressure" in df.columns:
    num_cols.append("pulse_pressure")

corr = df[num_cols].corr()
fig = px.imshow(corr, text_auto=".2f", aspect="auto",
                color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="Patient numeric feature correlations (no risk_level encoding)")
fig.show()

# Matplotlib heatmap for static report screenshots
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_title("Patient numeric correlations")
plt.tight_layout()
plt.show()

## Disclaimer

Patterns here reflect **synthetic labelling rules** used in Phase 1
(age, vitals, chronic disease, emergency flag, prior admissions).
They are intentionally visible for teaching — not evidence of real clinical risk models.